In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt

warnings.filterwarnings("ignore")

# ── Paths & config ───────────────────────────────────────────────
BASE_DIR   = "/home/am334/link_am334/praki/cmip6_data"

MODELS     = [
    "ACCESS-ESM1-5","ACCESS-CM2","CESM2","GFDL-ESM4","FGOALS-g3",
    "MRI-ESM2-0","MIROC6","CanESM5","GISS-E2-1-G","NorESM2-LM","NorESM2-MM",
    "HadGEM3-GC31-LL","UKESM1-1-LL","CMCC-ESM2","HadGEM3-GC31-MM",
    "MPI-ESM1-2-HR","INM-CM4-8","CanESM5-CanOE","GFDL-CM4","CAS-ESM2-0","IPSL-CM6A-LR"
]

SCEN_HIST  = "historical"
SCEN_FUT   = "ssp585"
LAT_TARGET = 26.5

# Output
OUT_DIR = os.path.join(BASE_DIR, "_mmm_timeseries_lpf10")
os.makedirs(OUT_DIR, exist_ok=True)

# which models use msftyz instead of msftmz
YZ_MODELS = {
    "CIESM","GFDL-CM4","GFDL-ESM4","HadGEM3-GC31-LL","CMCC-ESM2",
    "CMCC-CM2-HR4","CNRM-CM6-1-HR","EC-Earth3","HadGEM3-GC31-MM",
    "IPSL-CM6A-LR","IPSL-CM6A-MR1","UKESM1-1-LL"
}
# some models index the Atlantic basin differently
BASIN_IDX1_MODELS = {
    "MPI-ESM1-2-HR","CNRM-CM6-1-HR","EC-Earth3",
    "IPSL-CM6A-MR1","IPSL-CM6A-LR","E3SM-1-0"
}

# ── Helpers ──────────────────────────────────────────────────────
def butter_lpf_sos(cutoff_cpy=1/10, fs=1.0, order=5):
    """Return SOS for low-pass Butterworth with cutoff (cycles/year)."""
    nyq = 0.5 * fs
    wn = cutoff_cpy / nyq
    return butter(order, wn, btype="low", output="sos")

def lpf_1d_annual(y, valid_mask=None, cutoff_cpy=1/10, order=5):
    """
    Zero-phase LPF with Butterworth. Handles NaNs by linear interp
    over finite spans; leading/trailing NaNs remain NaN.
    """
    y = np.asarray(y, float)
    if valid_mask is None:
        valid_mask = np.isfinite(y)
    if valid_mask.sum() < 3:
        return np.full_like(y, np.nan)

    # Interpolate internal NaNs to allow filtering
    x = np.arange(y.size)
    yi = y.copy()
    # Preserve edges: only interpolate where we have finite endpoints
    if valid_mask.any():
        first = np.argmax(valid_mask)
        last  = len(valid_mask) - 1 - np.argmax(valid_mask[::-1])
        xi = x[valid_mask]
        yi_inside = np.interp(x[first:last+1], xi, y[valid_mask])
        yi[first:last+1] = yi_inside

    sos = butter_lpf_sos(cutoff_cpy=cutoff_cpy, fs=1.0, order=order)
    yf = sosfiltfilt(sos, yi)  # zero-phase
    # put NaNs back where no data originally
    yf[~valid_mask] = np.nan
    return yf

def prepare_da(ds, var, model):
    """Harmonize dims, pick Atlantic, standardize vertical & lat, and regrid."""
    da = ds[var]

    # 3-basin?
    if "3basin" in da.dims:
        da = da.rename({"3basin": "basin"}).isel(basin=1)
    if "basin" in da.dims:
        idx = 1 if model in BASIN_IDX1_MODELS else 0
        da = da.isel(basin=idx)

    # unify vertical
    if "depth" in da.dims and "lev" not in da.dims:
        da = da.rename(depth="lev")
    if "olevel" in da.dims and "lev" not in da.dims:
        da = da.rename(olevel="lev")

    # convert cm -> m if needed
    lev_units = str(da.lev.attrs.get("units", "")).lower()
    if lev_units in ("cm", "centimeter", "centimeters"):
        da = da.assign_coords(lev=da.lev / 100.0)

    # special-case IPSL-CM6A-LR staggered dims
    if model == "IPSL-CM6A-LR":
        if "x" in da.dims and da.sizes["x"] == 1:
            da = da.isel(x=0)
        nav = ds.get("nav_lat")
        if nav is not None:
            if "x" in nav.dims and nav.sizes["x"] == 1:
                nav = nav.isel(x=0)
            nav_vals = nav.values.copy()
            nav_vals[-1] = 90.0
            if "y" in da.dims:
                da = da.rename({"y": "rlat"})
            da = da.assign_coords(rlat=("rlat", nav_vals))
    else:
        for old in ("y","nav_lat","lat","latitude"):
            if old in da.dims:
                da = da.rename({old: "rlat"})
                break

    # restrict & interp to regular grid
    da = da.sel(lev=slice(0, 2500))
    da = da.interp(
        lev=np.arange(0, 2501, 100),
        rlat=np.arange(-30, 70.1, 2.5),
        method="linear"
    ).fillna(0)
    return da

def to_sv(da_1d):
    """Try to convert to Sv if units suggest kg/s; otherwise leave as-is."""
    units = str(da_1d.attrs.get("units", "")).lower()
    if "sv" in units:
        return da_1d
    return da_1d / 1e9

def subset_years(da, start_year, end_year):
    """Keep only start_year ≤ time.year ≤ end_year for any CF calendar."""
    year = da["time"].dt.year
    return da.where((year >= start_year) & (year <= end_year), drop=True)

def load_member_scenario_series(model, scenario, member, var):
    """
    Load monthly AMOC@26.5N (max over depth) for a member & scenario (no LPF).
    """
    var_dir = os.path.join(BASE_DIR, model, scenario, member, var)
    nc_files = sorted(glob.glob(os.path.join(var_dir, "*.nc")))
    if not nc_files:
        return None

    das = []
    for fn in nc_files:
        try:
            with xr.open_dataset(fn, use_cftime=True) as ds:
                if var in ds:
                    das.append(prepare_da(ds, var, model))
        except Exception as e:
            print(f"[err] {model}/{scenario}/{member} → {e}")

    if not das:
        return None

    moc = xr.concat(das, dim="time").sortby("time")
    # drop duplicate timestamps if any
    _, uniq_idx = np.unique(moc["time"].values, return_index=True)
    moc = moc.isel(time=np.sort(uniq_idx))

    # Historical through 2014; future through **2030** as requested
    if scenario == "historical":
        moc = subset_years(moc, 1850, 2014)
    else:
        moc = subset_years(moc, 2015, 2030)

    if moc.sizes.get("time", 0) == 0:
        return None

    amoc = moc.sel(rlat=LAT_TARGET, method="nearest").max(dim="lev")  # 1D monthly
    amoc = to_sv(amoc)
    return amoc

def stitch_hist_ssp(am_hist, am_ssp):
    """Concatenate historical ≤2014 with SSP ≥2015."""
    if am_hist is None and am_ssp is None:
        return None
    if am_hist is None:
        return am_ssp
    if am_ssp is None:
        return am_hist
    out = xr.concat([am_hist, am_ssp], dim="time").sortby("time")
    _, idx = np.unique(out["time"].values, return_index=True)
    return out.isel(time=np.sort(idx))

def monthly_to_annual(da_monthly):
    """Monthly 1D → annual mean with coord 'year'."""
    ann = da_monthly.groupby("time.year").mean("time")
    return ann.rename(year="year")

# ── Load & build member series ───────────────────────────────────
per_model_member_series = {}  # model -> list[pd.Series] (annual Sv)

for model in MODELS:
    var = "msftyz" if model in YZ_MODELS else "msftmz"
    hist_dir = os.path.join(BASE_DIR, model, SCEN_HIST)
    fut_dir  = os.path.join(BASE_DIR, model, SCEN_FUT)
    if not (os.path.isdir(hist_dir) and os.path.isdir(fut_dir)):
        print(f"[skip] {model}: missing {SCEN_HIST} or {SCEN_FUT} dir")
        continue

    hist_members = {os.path.basename(p) for p in glob.glob(os.path.join(hist_dir, "r*i*p*f*"))}
    fut_members  = {os.path.basename(p) for p in glob.glob(os.path.join(fut_dir,  "r*i*p*f*"))}
    common_members = sorted(hist_members & fut_members)
    if not common_members:
        print(f"[skip] {model}: no common members in {SCEN_HIST} & {SCEN_FUT}")
        continue

    print(f"[{model}] common members: {len(common_members)}")
    mm_list = []

    for member in common_members:
        am_hist = load_member_scenario_series(model, SCEN_HIST, member, var)
        am_ssp  = load_member_scenario_series(model, SCEN_FUT,  member, var)
        am = stitch_hist_ssp(am_hist, am_ssp)
        if am is None or am.sizes.get("time", 0) == 0:
            continue

        ann = monthly_to_annual(am)  # Sv, annual
        s = pd.Series(ann.values, index=ann.year.values.astype(int))  # 1850..2030 subset later
        mm_list.append(s)

    if mm_list:
        per_model_member_series[model] = mm_list

if not per_model_member_series:
    raise RuntimeError("No stitched member series found with both historical & ssp585.")

# ── Align, LPF(10yr) per member, then member-mean per model ─────
YEARS = np.arange(1900, 2031, dtype=int)  # inclusive 1900–2030
cutoff_cpy = 1/10  # 10-year LPF

model_means = {}  # model -> pd.Series (LPF(member) -> member-mean)
for model, members in per_model_member_series.items():
    aligned = [m.reindex(YEARS) for m in members]

    # Filter each member series
    filt_members = []
    for s in aligned:
        y = s.values
        m = np.isfinite(y)
        yf = lpf_1d_annual(y, valid_mask=m, cutoff_cpy=cutoff_cpy, order=5)
        filt_members.append(pd.Series(yf, index=YEARS))

    # Member-mean (across filtered members)
    if len(filt_members) == 1:
        model_means[model] = filt_members[0]
    else:
        dfm = pd.concat(filt_members, axis=1)
        model_means[model] = dfm.mean(axis=1, skipna=True)

# ── Multi-model mean (mean over models of member-means) ─────────
if not model_means:
    raise RuntimeError("No model means assembled after filtering.")

df_models = pd.DataFrame({mdl: s for mdl, s in model_means.items()}, index=YEARS)
mmm = df_models.mean(axis=1, skipna=True)  # MMM (Sv)

# ── Save CSVs ────────────────────────────────────────────────────
csv_model_means = os.path.join(OUT_DIR, "AMOC26N_model_membermeans_LPF10_1900-2030.csv")
csv_mmm         = os.path.join(OUT_DIR, "AMOC26N_MMM_LPF10_1900-2030.csv")
df_models.to_csv(csv_model_means, float_format="%.6f")
mmm.to_frame("MMM_Sv").to_csv(csv_mmm, float_format="%.6f")
print(f"[✓] saved {csv_model_means}")
print(f"[✓] saved {csv_mmm}")

# ── Plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10.5, 5.2))
# Light gray model means (optional: comment out if you only want MMM)
for mdl in df_models.columns:
    ax.plot(df_models.index, df_models[mdl].values, lw=1.0, alpha=0.35, label=None)

# Bold MMM
ax.plot(mmm.index, mmm.values, lw=2.4, label="Multi-model mean (LPF10→member-mean→model-mean)", zorder=5)

ax.set_xlim(1900, 2030)
ax.set_xlabel("Year")
ax.set_ylabel("AMOC at 26.5°N (Sv)")
ax.set_title("CMIP6 AMOC(26.5°N) Multi-Model Mean (members LPF10 first) — 1900–2030")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=9, loc="best")

plt.tight_layout()
out_png = os.path.join(OUT_DIR, "AMOC26N_MMM_LPF10_1900-2030.png")
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.close()
print(f"[✓] saved figure → {out_png}")


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import ConnectionPatch

warnings.filterwarnings("ignore")

# ── Paths & config ───────────────────────────────────────────────
BASE_DIR = "/home/am334/link_am334/praki/cmip6_data"
FIGS_DIR = "/home/am334/link_am334/figs"
OUT_DIR  = os.path.join(BASE_DIR, "_mmm_timeseries_lpf10_with_insets_anom")
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

MODELS = [
    "ACCESS-ESM1-5","ACCESS-CM2","CESM2","GFDL-ESM4","FGOALS-g3",
    "MRI-ESM2-0","MIROC6","CanESM5","GISS-E2-1-G","NorESM2-LM","NorESM2-MM",
    "HadGEM3-GC31-LL","UKESM1-1-LL","CMCC-ESM2","HadGEM3-GC31-MM",
    "MPI-ESM1-2-HR","INM-CM4-8","CanESM5-CanOE","GFDL-CM4","CAS-ESM2-0","IPSL-CM6A-LR"
]



#MODELS = [
#"NorESM2-LM","CMCC-ESM2", "UKESM1-1-LL", "GFDL-ESM4"
#]



SCEN_HIST  = "historical"
SCEN_FUT   = "ssp585"
SCEN_PICTL = "piControl"

LAT_TARGET = 26.5
YEARS = np.arange(1900, 2031, dtype=int)     # 1900–2030 inclusive

# Snapshot windows (inclusive)
SNAP_WINDOWS = {
    1985: (1983, 1987),
    2030: (2028, 2030),
}

# LPF (10-year)
CUTOFF_CPY = 1/10
LPF_ORDER  = 5

# Regular grids for interpolation
GRID_LEV  = np.arange(0, 2501, 100)       # m
GRID_RLAT = np.arange(-30, 70.1, 2.5)     # deg

# Which models use msftyz instead of msftmz
YZ_MODELS = {
    "CIESM","GFDL-CM4","GFDL-ESM4","HadGEM3-GC31-LL","CMCC-ESM2",
    "CMCC-CM2-HR4","CNRM-CM6-1-HR","EC-Earth3","HadGEM3-GC31-MM",
    "IPSL-CM6A-LR","IPSL-CM6A-MR1","UKESM1-1-LL"
}
# Models with basin index=1 for Atlantic
BASIN_IDX1_MODELS = {
    "MPI-ESM1-2-HR","CNRM-CM6-1-HR","EC-Earth3",
    "IPSL-CM6A-MR1","IPSL-CM6A-LR","E3SM-1-0"
}

# ── Helpers ──────────────────────────────────────────────────────
def butter_lpf_sos(cutoff_cpy=CUTOFF_CPY, fs=1.0, order=LPF_ORDER):
    nyq = 0.5 * fs
    wn = cutoff_cpy / nyq
    return butter(order, wn, btype="low", output="sos")

def lpf_1d_annual(y, valid_mask=None, cutoff_cpy=CUTOFF_CPY, order=LPF_ORDER):
    y = np.asarray(y, float)
    if valid_mask is None:
        valid_mask = np.isfinite(y)
    if valid_mask.sum() < 3:
        return np.full_like(y, np.nan)
    x = np.arange(y.size)
    yi = y.copy()
    if valid_mask.any():
        first = np.argmax(valid_mask)
        last  = len(valid_mask) - 1 - np.argmax(valid_mask[::-1])
        xi = x[valid_mask]
        yi[first:last+1] = np.interp(x[first:last+1], xi, y[valid_mask])
    sos = butter_lpf_sos(cutoff_cpy=cutoff_cpy, fs=1.0, order=order)
    yf  = sosfiltfilt(sos, yi)
    yf[~valid_mask] = np.nan
    return yf

def to_sv(da):
    units = str(da.attrs.get("units", "")).lower()
    if "sv" in units:
        return da
    return da / 1e9

def prepare_da(ds, var, model):
    da = ds[var]
    # basin dimension
    if "3basin" in da.dims:
        da = da.rename({"3basin": "basin"}).isel(basin=1)
    if "basin" in da.dims:
        idx = 1 if model in BASIN_IDX1_MODELS else 0
        da = da.isel(basin=idx)
    # vertical name
    if "depth" in da.dims and "lev" not in da.dims:
        da = da.rename(depth="lev")
    if "olevel" in da.dims and "lev" not in da.dims:
        da = da.rename(olevel="lev")
    # cm -> m
    lev_units = str(da.lev.attrs.get("units", "")).lower() if "lev" in da.dims else ""
    if lev_units in ("cm", "centimeter", "centimeters"):
        da = da.assign_coords(lev=da.lev / 100.0)
    # latitude name
    if str(model) == "IPSL-CM6A-LR":
        if "x" in da.dims and da.sizes.get("x", 1) == 1:
            da = da.isel(x=0)
        nav = ds.get("nav_lat")
        if nav is not None:
            if "x" in nav.dims and nav.sizes.get("x", 1) == 1:
                nav = nav.isel(x=0)
            nav_vals = nav.values.copy()
            nav_vals[-1] = 90.0
            if "y" in da.dims:
                da = da.rename({"y": "rlat"})
            da = da.assign_coords(rlat=("rlat", nav_vals))
    else:
        for old in ("y","nav_lat","lat","latitude"):
            if old in da.dims:
                da = da.rename({old: "rlat"})
                break
    # regrid
    if "lev" in da.dims:
        da = da.sel(lev=slice(0, 2500))
    da = da.interp(
        lev=GRID_LEV if "lev" in da.dims else GRID_LEV,
        rlat=GRID_RLAT,
        method="linear"
    ).fillna(0)
    return to_sv(da)

def subset_years(da, start_year, end_year):
    year = da["time"].dt.year
    return da.where((year >= start_year) & (year <= end_year), drop=True)

def open_concat(model, scenario, member, var):
    var_dir = os.path.join(BASE_DIR, model, scenario, member, var)
    files = sorted(glob.glob(os.path.join(var_dir, "*.nc")))
    if not files:
        return None
    das = []
    for fn in files:
        try:
            with xr.open_dataset(fn, use_cftime=True) as ds:
                if var in ds:
                    das.append(prepare_da(ds, var, model))
        except Exception as e:
            print(f"[err] {model}/{scenario}/{member}: {e}")
    if not das:
        return None
    da = xr.concat(das, dim="time").sortby("time")
    _, uniq_idx = np.unique(da["time"].values, return_index=True)
    return da.isel(time=np.sort(uniq_idx))

def load_member_series_26N(model, scenario, member, var):
    da = open_concat(model, scenario, member, var)
    if da is None or da.sizes.get("time", 0) == 0:
        return None
    if scenario == SCEN_HIST:
        da = subset_years(da, 1850, 2014)
    elif scenario == SCEN_FUT:
        da = subset_years(da, 2015, 2030)
    if da.sizes.get("time", 0) == 0:
        return None
    amoc = da.sel(rlat=LAT_TARGET, method="nearest").max(dim="lev")
    return amoc  # monthly Sv

def load_member_field(model, scenario, member, var):
    da = open_concat(model, scenario, member, var)
    if da is None or da.sizes.get("time", 0) == 0:
        return None
    if scenario == SCEN_HIST:
        da = subset_years(da, 1850, 2014)
    elif scenario == SCEN_FUT:
        da = subset_years(da, 2015, 2030)
    return da  # monthly Sv

def stitch_hist_ssp(da_hist, da_ssp):
    if da_hist is None and da_ssp is None:
        return None
    if da_hist is None:
        return da_ssp
    if da_ssp is None:
        return da_hist
    out = xr.concat([da_hist, da_ssp], dim="time").sortby("time")
    _, idx = np.unique(out["time"].values, return_index=True)
    return out.isel(time=np.sort(idx))

def monthly_to_annual_1d(da):
    ann = da.groupby("time.year").mean("time")
    return ann.rename(year="year")  # -> 1D (year,)

def monthly_to_annual_2d(da):
    ann = da.groupby("time.year").mean("time")
    return ann.rename(year="year")  # -> 3D (year, lev, rlat)

# ────────────────────────────────────────────────────────────────
#                 █ 1) COMPUTE & SAVE DATA FIRST █
# ────────────────────────────────────────────────────────────────


ESSENTIAL_COORDS = {"time", "year", "lev", "rlat"}

def clean_coords(da: xr.DataArray) -> xr.DataArray:
    """
    Remove auxiliary coords (e.g., 'sector', 'lat_aux_grid', etc.) so that
    concatenation across models doesn't fail. Preserves dims and essential coords.
    """
    # some coords may share names with dims; don't drop dims
    to_drop = [c for c in da.coords if (c not in ESSENTIAL_COORDS and c not in da.dims)]
    if to_drop:
        da = da.reset_coords(names=to_drop, drop=True)
    return da

    
# 1A) Per-model piControl baselines
def model_picontrol_mean_26N(model, var):
    pic_dir = os.path.join(BASE_DIR, model, SCEN_PICTL)
    if not os.path.isdir(pic_dir):
        return np.nan
    vals = []
    for member in sorted(os.listdir(pic_dir)):
        if not member.startswith("r"):  # skip non-member dirs
            continue
        da = load_member_field(model, SCEN_PICTL, member, var)
        if da is None:
            continue
        # annual, select 26.5N, max over depth → time series, then time mean
        ann = monthly_to_annual_2d(da)  # (year, lev, rlat)
        ts  = ann.sel(rlat=LAT_TARGET, method="nearest").max(dim="lev")
        if ts.size == 0:
            continue
        vals.append(float(ts.mean("year", skipna=True).values))
    if not vals:
        return np.nan
    return float(np.nanmean(vals))

def model_picontrol_mean_field(model, var):
    pic_dir = os.path.join(BASE_DIR, model, SCEN_PICTL)
    if not os.path.isdir(pic_dir):
        return None
    fields = []
    for member in sorted(os.listdir(pic_dir)):
        if not member.startswith("r"):
            continue
        da = load_member_field(model, SCEN_PICTL, member, var)
        if da is None:
            continue
        fields.append(da.mean("time", skipna=True))
    if not fields:
        return None

    out = xr.concat(fields, dim="stacked_member", coords="minimal", compat="override", join="outer").mean("stacked_member", skipna=True)
    out = clean_coords(out)
    return out
    
    #return xr.concat(fields, dim="stacked_member").mean("stacked_member", skipna=True)

pi_baseline_26N = {}   # model -> scalar Sv
pi_field_mean   = {}   # model -> (lev, rlat)

for model in MODELS:
    var = "msftyz" if model in YZ_MODELS else "msftmz"
    b26 = model_picontrol_mean_26N(model, var)
    if np.isfinite(b26):
        pi_baseline_26N[model] = b26
    mf  = model_picontrol_mean_field(model, var)
    if mf is not None:
        pi_field_mean[model] = mf

# Save piControl baselines
pd.Series(pi_baseline_26N, name="piControl_mean_Sv").to_csv(
    os.path.join(OUT_DIR, "piControl_baseline_AMOC26N_by_model.csv"),
    float_format="%.6f"
)
# Save fields
for model, fld in pi_field_mean.items():
    fld.to_dataset(name="psi_pi_mean").to_netcdf(os.path.join(OUT_DIR, f"piControl_mean_field_{model}.nc"))

# 1B) Per-model member series (historical+ssp585), annual → anomaly vs piControl → LPF10 → member-mean
per_model_membermeans_anom = {}  # model -> pd.Series (YEARS)
for model in MODELS:
    var = "msftyz" if model in YZ_MODELS else "msftmz"

    hist_dir = os.path.join(BASE_DIR, model, SCEN_HIST)
    fut_dir  = os.path.join(BASE_DIR, model, SCEN_FUT)
    if not (os.path.isdir(hist_dir) and os.path.isdir(fut_dir)):
        continue

    hist_members = {os.path.basename(p) for p in glob.glob(os.path.join(hist_dir, "r*i*p*f*"))}
    fut_members  = {os.path.basename(p) for p in glob.glob(os.path.join(fut_dir,  "r*i*p*f*"))}
    common_members = sorted(hist_members & fut_members)
    if not common_members:
        continue

    baseline = pi_baseline_26N.get(model, np.nan)
    mmembers = []
    for member in common_members:
        da_h = load_member_series_26N(model, SCEN_HIST, member, var)
        da_s = load_member_series_26N(model, SCEN_FUT,  member, var)
        da   = stitch_hist_ssp(da_h, da_s)
        if da is None or da.sizes.get("time", 0) == 0:
            continue
        ann = monthly_to_annual_1d(da)            # (year,)
        s   = pd.Series(ann.values, index=ann.year.values.astype(int))
        s   = s.reindex(YEARS)

        # anomaly vs model's piControl mean
        if np.isfinite(baseline):
            s = s - baseline
        # LPF-10y
        y = s.values
        m = np.isfinite(y)
        yf = lpf_1d_annual(y, m)
        mmembers.append(pd.Series(yf, index=YEARS))

    if mmembers:
        dfm = pd.concat(mmembers, axis=1)
        per_model_membermeans_anom[model] = dfm.mean(axis=1, skipna=True)

if not per_model_membermeans_anom:
    raise RuntimeError("No per-model member-mean anomaly series assembled.")

# Multi-model mean anomaly time series
df_models_anom = pd.DataFrame(per_model_membermeans_anom, index=YEARS)
mmm_anom = df_models_anom.mean(axis=1, skipna=True)

# Save anomaly time series
df_models_anom.to_csv(os.path.join(OUT_DIR, "AMOC26N_model_membermeans_LPF10_ANOMvsPI_1900-2030.csv"), float_format="%.6f")
mmm_anom.to_frame("MMM_anom_Sv").to_csv(os.path.join(OUT_DIR, "AMOC26N_MMM_LPF10_ANOMvsPI_1900-2030.csv"), float_format="%.6f")

# 1C) Build MMM Δψ (lev×lat) anomaly fields for snapshots
def model_snapshot_membermean_field(model, years_window, var):
    hist_dir = os.path.join(BASE_DIR, model, SCEN_HIST)
    fut_dir  = os.path.join(BASE_DIR, model, SCEN_FUT)
    if not (os.path.isdir(hist_dir) and os.path.isdir(fut_dir)):
        return None
    hist_members = {os.path.basename(p) for p in glob.glob(os.path.join(hist_dir, "r*i*p*f*"))}
    fut_members  = {os.path.basename(p) for p in glob.glob(os.path.join(fut_dir,  "r*i*p*f*"))}
    common_members = sorted(hist_members & fut_members)
    if not common_members:
        return None

    y0, y1 = years_window
    fields = []
    for member in common_members:
        da_h = load_member_field(model, SCEN_HIST, member, var)
        da_s = load_member_field(model, SCEN_FUT,  member, var)
        da   = stitch_hist_ssp(da_h, da_s)
        if da is None or da.sizes.get("time", 0) == 0:
            continue
        ann = monthly_to_annual_2d(da)  # (year, lev, rlat)
        sub = ann.sel(year=slice(y0, y1))
        if sub.sizes.get("year", 0) == 0:
            continue
        fields.append(sub.mean("year", skipna=True))
    if not fields:
        return None

    out = xr.concat(fields, dim="stacked_member", coords="minimal", compat="override", join="outer").mean("stacked_member", skipna=True)
    out = clean_coords(out)  # << strip aux coords her

    
    #return xr.concat(fields, dim="stacked_member").mean("stacked_member", skipna=True)
    return out

# Keep only the coords we actually need for plotting/combining



def build_mmm_field_anomaly(years_window):
    deltas = []
    for model in MODELS:
        print(f"--[{model}]")
        var = "msftyz" if model in YZ_MODELS else "msftmz"
        snap = model_snapshot_membermean_field(model, years_window, var)
        picm = pi_field_mean.get(model)
        if snap is None or picm is None:
            continue
        delta = (snap - picm).where(np.isfinite(snap - picm))
        # limit for cleaner insets
        delta = delta.sel(rlat=slice(0, 60), lev=slice(0, 2000))
        deltas.append(clean_coords(delta))
    if not deltas:
        return None
    return xr.concat(deltas, dim="stacked_model").mean("stacked_model", skipna=True)

anom_fields = {}
for yr, win in SNAP_WINDOWS.items():
    fld = build_mmm_field_anomaly(win)
    if fld is not None:
        anom_fields[yr] = fld
        # save to netcdf
        fld.to_dataset(name="dpsi").to_netcdf(
            os.path.join(OUT_DIR, f"MMM_AMOC_anomaly_vs_piControl_{yr}_{win[0]}-{win[1]}.nc")
        )

# ────────────────────────────────────────────────────────────────
#                     █ 2) PLOTTING SECTION █
# ────────────────────────────────────────────────────────────────

def plot_ts(df_models, mmm, out_png):
    fig, ax = plt.subplots(figsize=(10.5, 5.0))
    for mdl in df_models.columns:
        ax.plot(df_models.index, df_models[mdl].values, lw=0.9, alpha=0.28)
    ax.plot(mmm.index, mmm.values, lw=2.4, label="MMM (LPF10, anomaly vs piControl)", zorder=5)
    ax.axhline(0, lw=1.0, color="k", alpha=0.6)
    ax.set_xlim(1900, 2030)
    ax.set_xlabel("Year")
    ax.set_ylabel("AMOC@26.5°N anomaly (Sv)")
    ax.set_title("CMIP6 AMOC(26.5°N) — Multi-Model Mean Anomaly vs piControl (1900–2030)")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=9, loc="best")
    plt.tight_layout()
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.close()


In [ ]:
import numpy as np
import xarray as xr

# Keep only the coords we need so xarray concat doesn't blow up on extra coords like 'sector'
ESSENTIAL_COORDS = {"time", "year", "lev", "rlat", "basin"}

def to_sv_safe(da: xr.DataArray) -> xr.DataArray:
    """Ensure streamfunction is in Sverdrups (Sv)."""
    units = str(da.attrs.get("units", "")).lower()
    if "sv" in units:
        return da
    out = da / 1e9
    out.attrs["units"] = "Sv"
    return out

def clean_coords(da: xr.DataArray) -> xr.DataArray:
    """Drop non-essential coords (e.g., 'sector') while preserving dims."""
    to_drop = [c for c in da.coords if (c not in ESSENTIAL_COORDS and c not in da.dims)]
    if to_drop:
        da = da.reset_coords(names=to_drop, drop=True)
    return da

def build_mmm_pi_field(
    pi_field_mean: dict,
    out_nc: str | None = None,
    target_rlat: np.ndarray | None = None,
    target_lev:  np.ndarray | None = None,
    rlat_slice: tuple[float,float] = (0.0, 60.0),
    lev_slice:  tuple[float,float] = (0.0, 2000.0),
    name: str = "psi_pi_MMM"
) -> xr.DataArray:
    """
    Compute MMM piControl streamfunction (Sv) on a common (lev × rlat) grid,
    trimmed to the plotting domain. Optionally save to NetCDF.

    Parameters
    ----------
    pi_field_mean : dict[str, xr.DataArray]
        Per-model piControl mean fields (lev × rlat). Values may have extra coords.
    out_nc : str | None
        If provided, save as NetCDF to this path.
    target_rlat : array | None
        If provided, interpolate MMM to these latitudes (deg).
    target_lev : array | None
        If provided, interpolate MMM to these depths (m).
    rlat_slice : (float,float)
        Latitude slice to keep after (re)gridding.
    lev_slice : (float,float)
        Depth slice to keep after (re)gridding.
    name : str
        Variable name in the saved dataset.

    Returns
    -------
    xr.DataArray  (lev, rlat) in Sv
    """
    if not pi_field_mean:
        raise ValueError("pi_field_mean is empty; build per-model piControl means first.")

    # 1) Collect, convert to Sv, and sanitize coords
    fields = []
    for mdl, fld in pi_field_mean.items():
        if fld is None or not isinstance(fld, xr.DataArray):
            continue
        #f = to_sv_safe(fld)
        f = clean_coords(fld)
        # ensure dims order contains lev and rlat
        if "lev" not in f.dims or "rlat" not in f.dims:
            raise ValueError(f"{mdl}: expected dims to include 'lev' and 'rlat', got {f.dims}")
        fields.append(f)

    if not fields:
        raise ValueError("No valid piControl fields found in pi_field_mean.")

    # 2) (Optional) interpolate each field to a target grid BEFORE concat (helps alignment)
    if target_rlat is not None or target_lev is not None:
        _interp_fields = []
        for f in fields:
            f2 = f
            if target_lev is not None:
                f2 = f2.interp(lev=xr.DataArray(target_lev, dims="lev"))
            if target_rlat is not None:
                f2 = f2.interp(rlat=xr.DataArray(target_rlat, dims="rlat"))
            _interp_fields.append(f2)
        fields = _interp_fields

    # 3) Concatenate across models using relaxed coord rules
    mmm = xr.concat(fields, dim="stacked_model", coords="minimal", compat="override", join="outer") \
            .mean("stacked_model", skipna=True)

    # 4) Trim to plotting domain and ensure correct units
    mmm = mmm.sel(rlat=slice(*rlat_slice), lev=slice(*lev_slice))
    mmm.attrs["units"] = "Sv"

    # 5) Optionally save
    if out_nc is not None:
        mmm.to_dataset(name=name).to_netcdf(out_nc)

    return mmm
# Choose your plotting grid (match what you use in insets)
TARGET_RLAT = np.arange(0.0, 60.01, 2.5)   # 0–60°N
TARGET_LEV  = np.arange(0.0, 2000.01, 100) # 0–2000 m

OUT_PI_MMM = os.path.join(OUT_DIR, "piControl_MMM_streamfunction_0-60N_0-2000m.nc")

mmm_pi_field = build_mmm_pi_field(
    pi_field_mean=pi_field_mean,
    out_nc=OUT_PI_MMM,
    target_rlat=TARGET_RLAT,
    target_lev=TARGET_LEV,
    rlat_slice=(0.0, 60.0),
    lev_slice=(0.0, 2000.0),
    name="psi_pi_MMM"
)


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

def load_proxy_series_nc(nc_path: str,
                         factor: float = 3.8,
                         baseline: tuple[int,int] = (1900, 1940)) -> pd.Series:
    """
    Load a 1D time series from a NetCDF file and return anomalies in Sv.
    - Multiplies by `factor` (default 3.8 Sv/K) to convert to Sv.
    - Anomalies are relative to the mean over `baseline` inclusive.
    - Returns a pandas Series indexed by year (int).
    """
    with xr.open_dataset(nc_path) as ds:
        # pick the first data variable robustly
        data_vars = list(ds.data_vars)
        if not data_vars:
            raise ValueError(f"No data variables found in {nc_path}")
        da = ds["AMOC_LPF10"]

        # get annual series
        if "year" in da.dims:
            years = da["year"].values.astype(int)
            vals = np.asarray(da.values).squeeze()
        elif "time" in da.dims:
            ann = da.groupby("time.year").mean("time")
            years = ann["year"].values.astype(int)
            vals = np.asarray(ann.values).squeeze()
        else:
            # fallback: try to find a coordinate that looks like years
            cand = next((c for c in da.coords if "year" in c.lower()), None)
            if cand is None:
                raise ValueError("No 'time' or 'year' dim/coord found.")
            years = da[cand].values.astype(int)
            vals = np.asarray(da.values).squeeze()

    s = pd.Series(vals, index=years).sort_index()
    # baseline anomaly (inclusive)
    base = float(s.loc[baseline[0]:baseline[1]].mean())
    s_anom = (s - base) * factor
    s_anom.name = f"Caesar/HadISST ×{factor:g} (Sv; base {baseline[0]}–{baseline[1]})"
    return s_anom
    
PROXY_NC = "/dat1/am334/figs/data/AMOC_Caesar_HadISST_LPF10.nc"
proxy_series = load_proxy_series_nc(PROXY_NC, factor=3.8, baseline=(1900, 1940)) # 3.8 from Cesear et al 2018

In [ ]:
from matplotlib.patches import ConnectionPatch, Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def to_sv_safe(da):
    """Ensure streamfunction is in Sverdrups (Sv)."""
    units = str(da.attrs.get("units", "")).lower()
    if "sv" in units:
        return da
    return da / 1e9


def plot_field(field, title, out_png, vmax=None, pi_field=None, pi_levels=None):
    """
    Standalone Δψ(lev×lat) plot with optional MMM piControl ψ contours.
    All values are assumed or converted to Sv.

    Args:
        field      (xr.DataArray): Δψ in Sv, dims (lev, rlat).
        title      (str): Figure title.
        out_png    (str): Output path for PNG.
        vmax       (float|None): Abs color limit for Δψ; auto if None.
        pi_field   (xr.DataArray|None): MMM piControl ψ (Sv), dims (lev, rlat) to overlay as contours.
        pi_levels  (list[float]|None): Contour levels for pi_control ψ; auto 5-Sv steps if None.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import xarray as xr

    if field is None:
        return

    # ── Units sanity ─────────────────────────────────────────────
    #field = to_sv_safe(field)
    if pi_field is not None:
        pi_field = to_sv_safe(pi_field)

    # ── Figure setup ─────────────────────────────────────────────
    fig = plt.figure(figsize=(4.8, 3.4), constrained_layout=True)
    ax  = fig.add_subplot(111)

    X, Y = np.meshgrid(field["rlat"].values, field["lev"].values)

    if vmax is None:
        vmax = np.nanmax(np.abs(field.values))
        vmax = float(np.ceil(vmax * 2) / 2.0) if np.isfinite(vmax) else 2.0

    # ── Background: Δψ anomaly ───────────────────────────────────
    pcm = ax.pcolormesh(X, Y, field.values, cmap="RdBu_r",
                        vmin=-vmax, vmax=vmax, shading="auto")

    # Zero-anomaly contour
    try:
        ax.contour(X, Y, field.values, levels=[0.0], colors="k", linewidths=0.7)
    except Exception:
        pass

    # ── Overlay MMM piControl ψ contours ─────────────────────────
    if pi_field is not None:
        pf = pi_field.interp(rlat=field.rlat, lev=field.lev)
        if pi_levels is None:
            pf_max = float(np.nanmax(pf.values)) if np.isfinite(np.nanmax(pf.values)) else 10.0
            top = max(10.0, float(np.ceil(pf_max / 5.0) * 5.0))
            pi_levels = np.arange(5.0, top + 0.1, 5.0)
        cs = ax.contour(X, Y, pf.values, levels=pi_levels,
                        colors="k", linewidths=0.7, linestyles=":")
        try:
            ax.clabel(cs, fmt="%.0f", inline=True, fontsize=7)
        except Exception:
            pass

    # ── Cosmetics ────────────────────────────────────────────────
    ax.set_xlim(0, 60)
    ax.set_ylim(2000, 0)
    ax.set_xlabel("Latitude (°N)", fontsize=9)
    ax.set_ylabel("Depth (m)", fontsize=9)
    ax.set_title(f"{title}\nΔψ (Sv)", fontsize=9)  # ← include units in title

    cb = fig.colorbar(pcm, ax=ax, orientation="vertical", fraction=0.05, pad=0.02)
    cb.set_label("Δψ (Sverdrups, Sv)", fontsize=8)
    cb.ax.tick_params(labelsize=7)

    fig.savefig(out_png, dpi=240, bbox_inches="tight")
    plt.close(fig)

from matplotlib.patches import ConnectionPatch, Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import matplotlib.pyplot as plt

def plot_combo(df_models, mmm, anom_fields, windows, out_png,
               pi_field=None, pi_levels=None,
               overlays: dict[str, pd.Series] | None = None):
    
    """
    Time series + small Δψ insets at the bottom.
    - Connectors originate from a tight MMM range box over [x0, x1].
    - Connectors are BEHIND inset contents (low zorder, drawn before pcolormesh).
    - Insets show a vertical line at 26°N and have compact colorbars.
    """
    fig, ax = plt.subplots(figsize=(16.0, 6.0))

    # Background lines and MMM
    for mdl in df_models.columns:
        ax.plot(df_models.index, df_models[mdl].values, lw=0.8, alpha=0.25)
    ax.plot(mmm.index, mmm.values, lw=2.4, label="MMM", zorder=5)
    #ax.axhline(0, lw=1.0, color="k", alpha=0.6)


    if overlays:
        for label, ser in overlays.items():
            # align to x-range; no interpolation (only plot existing years)
            ser_plot = ser[ser.index.intersection(mmm.index)]
            ax.plot(ser_plot.index, ser_plot.values, lw=1.8, alpha=0.95,
                    linestyle="-", label=label, zorder=6)


            
    ax.set_xlim(1900, 2030)
    ax.set_xlabel("Year")
    ax.set_ylabel("AMOC 26.5°N anomaly (Sv)")
    #ax.set_title("CMIP6 AMOC (26.5°N) Anomaly")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=9, loc="best")

    # Shared Δψ color range
    vmax = 2.0
    if anom_fields:
        vmaxs = [np.nanmax(np.abs(f.values)) for f in anom_fields.values() if f is not None]
        if vmaxs:
            vmax = float(np.ceil(np.nanmax(vmaxs) * 2) / 2.0)

    # Bottom, smaller insets: x0, y0, w, h (axes fraction)
    inset_specs = {
        1985: dict(bounds=(0.18, 0.08, 0.17, 0.17)),  # tighter & lower
        2030: dict(bounds=(0.62, 0.08, 0.17, 0.17)),
    }

    # Light bands for windows
    for _, (x0, x1) in windows.items():
        ax.axvspan(x0, x1, color="gray", alpha=0.08, lw=0, zorder=1)

    for yr, (x0, x1) in windows.items():
        # Tight MMM range box over [x0, x1]
        seg = mmm.loc[x0:x1]
        if seg.size and np.isfinite(seg.values).any():
            y_min = float(np.nanmin(seg.values))
            y_max = float(np.nanmax(seg.values))
        else:
            y_min, y_max = -0.2, 0.2

        pad = max(0.05, 0.05 * (y_max - y_min))
        y0_box, y1_box = y_min - pad, y_max + pad
        if not np.isfinite(y0_box) or not np.isfinite(y1_box) or y1_box <= y0_box:
            y0_box, y1_box = -0.2, 0.2

        # Draw the tight source box (behind)
        ax.add_patch(Rectangle((x0, y0_box), x1 - x0, y1_box - y0_box,
                               fill=False, lw=1.0, ls="--", ec="k", alpha=0.4, zorder=1))

        # Create the REAL inset (first), push it above connectors via high zorder
        iax = ax.inset_axes(inset_specs[yr]["bounds"])
        iax.set_zorder(30)             # ensure axes (and its children) render above connectors
        iax.set_facecolor("white")     # crisp background

        # Compute corners in inset axes coordinates
        tgt_corners = [(0.0, 0.0), (1.0, 0.0), (1.0, 1.0), (0.0, 1.0)]
        src_corners = [(x0, y0_box), (x1, y0_box), (x1, y1_box), (x0, y1_box)]

        # Draw connectors NOW with **low zorder** so they stay behind the inset contents
        for (sx, sy), (tx, ty) in zip(src_corners, tgt_corners):
            con = ConnectionPatch(
                xyA=(sx, sy), coordsA=ax.transData,
                xyB=(tx, ty), coordsB=iax.transAxes,
                arrowstyle="-", lw=1.0, alpha=0.40, color="gray",
                zorder=1, clip_on=False
            )
            fig.add_artist(con)

        # Fill the inset contents on top of the connectors
        fld = anom_fields.get(yr)
        if fld is not None:
            X, Y = np.meshgrid(fld["rlat"].values, fld["lev"].values)
            pcm = iax.pcolormesh(X, Y, fld.values, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto", zorder=35)

            # 0-anomaly contour
            #try:
            #    iax.contour(X, Y, fld.values, levels=[0.0], colors="k", linewidths=0.7, zorder=36)
            #except Exception:
            #    pass

            # piControl ψ contours
            if pi_field is not None:
                pf = pi_field.interp(rlat=fld.rlat, lev=fld.lev)
                if pi_levels is None:
                    pf_max = np.nanmax(pf.values)
                    top = max(10.0, float(np.ceil(pf_max / 5.0) * 5.0)) if np.isfinite(pf_max) else 10.0
                    levels = np.arange(5.0, top + 0.1, 5.0)
                else:
                    levels = pi_levels
                cs = iax.contour(X, Y, pf.values, levels=levels,
                                 colors="k", linewidths=0.7, linestyles=":", zorder=36)
                try:
                    iax.clabel(cs, fmt="%.0f", inline=True, fontsize=6)
                except Exception:
                    pass

            # 26°N vertical line
            iax.axvline(26.0, lw=1.0, color="k", alpha=0.9, linestyle="--", zorder=37)

            iax.set_xlim(0, 60)
            iax.set_ylim(2000, 0)
            iax.set_xticks([0, 30, 60]); iax.set_yticks([0, 1000, 2000])
            iax.tick_params(labelsize=6)
            iax.set_title(f"Δψ {x0}–{x1} (Sv)", fontsize=7)

            # Smaller, slightly higher colorbar
            cax = inset_axes(iax, width="50%", height="3%", loc="lower center", borderpad=1.5)
            cb = fig.colorbar(pcm, cax=cax, orientation="horizontal")
            cb.set_label("Δψ (Sv)", fontsize=6)
            cb.ax.tick_params(labelsize=5)
        else:
            iax.text(0.5, 0.5, f"No data {x0}–{x1}", ha="center", va="center", fontsize=7, zorder=35)
            iax.axis("off")

        # neat inset frame
        for sp in iax.spines.values():
            sp.set_linewidth(0.8)

    plt.savefig(out_png, dpi=260, bbox_inches="tight")
    plt.close(fig)





plot_ts(
    df_models=df_models_anom,
    mmm=mmm_anom,
    out_png=os.path.join(FIGS_DIR, "AMOC26N_MMM_LPF10_ANOMvsPI_1900-2030_fig_ts.png"),
)

if 1985 in anom_fields:
    plot_field(
        anom_fields[1985],
        title="Δψ vs piControl — MMM (1983–1987)",
        out_png=os.path.join(FIGS_DIR, "AMOC_MMM_dpsi_1985.png"),
    )

if 2030 in anom_fields:
    plot_field(
        anom_fields[2030],
        title="Δψ vs piControl — MMM (2028–2030)",
        out_png=os.path.join(FIGS_DIR, "AMOC_MMM_dpsi_2030.png"),
    )

# ── Combined figure with smaller insets & connector lines ───────
plot_combo(
    df_models=df_models_anom,
    mmm=mmm_anom,
    anom_fields=anom_fields,
    windows=SNAP_WINDOWS,
    out_png=os.path.join(FIGS_DIR, "AMOC_MMM_anom_with_insets_combo_plus_HadISST.png"),
    pi_field=mmm_pi_field,
    overlays={"Caesar et al": proxy_series},
)


print("[✓] Saved all figures to:", FIGS_DIR)
print("[✓] Saved data to:", OUT_DIR)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch, Rectangle, FancyArrowPatch

def plot_combo_external_insets_fullheight_onecb_vertical(
    df_models, mmm, anom_fields, windows, out_png,
    pi_field=None, pi_levels=None, overlays=None,
    save_individual=True,
    rebase=True,
    baseline_years=(1900, 1940)
):
    # === Helpers ===
    def _to_years(index):
        try:
            years = pd.Index(index)
            if np.issubdtype(years.dtype, np.integer):
                return years.values.astype(int)
            return pd.to_datetime(years).year.values.astype(int)
        except Exception:
            return np.asarray(index, dtype=int)

    def _baseline_mask(years, y0, y1):
        return (years >= y0) & (years <= y1)

    def _rebase_series_like(series, y0, y1):
        yrs = _to_years(series.index)
        msk = _baseline_mask(yrs, y0, y1)
        if not np.any(msk):
            return series
        base = np.nanmean(series.loc[series.index[msk]].values)
        return series - base

    def _rebase_df_like(df, y0, y1):
        yrs = _to_years(df.index)
        msk = _baseline_mask(yrs, y0, y1)
        if not np.any(msk):
            return df
        dfb = df.copy()
        base = dfb.loc[dfb.index[msk]].mean(axis=0, skipna=True)
        dfb = dfb.subtract(base, axis=1)
        return dfb

    def _draw_amoc_arrows(iax):
        """
        Two horizontal schematic AMOC arrows:
          Upper: southward → northward surface branch, left to right, shallow depth
          Lower: northward → southward deep return branch, right to left, deep
        """
        kw = dict(
            arrowstyle     = "-|>",
            mutation_scale = 16,
            lw             = 1.8,
            color          = "k",
            alpha          = 0.75,
            zorder         = 9,
            clip_on        = True,
            transform      = iax.transData,
        )

        # Upper branch: northward surface flow, left → right at ~500 m depth
        iax.add_patch(FancyArrowPatch(
            posA=(5, 400), posB=(52, 400),
            connectionstyle="arc3,rad=0.0",   # straight horizontal
            **kw
        ))

        # Lower branch: southward deep return flow, right → left at ~1600 m depth
        iax.add_patch(FancyArrowPatch(
            posA=(52, 1300), posB=(5, 1300),
            connectionstyle="arc3,rad=0.0",   # straight horizontal
            **kw
        ))

    # === Font/line parameters ===
    AXIS_LABEL_FS       = 16
    TICK_LABEL_FS_MAIN  = 16
    TICK_LABEL_FS_INSET = 15
    INSET_LABEL_FS      = 16
    INSET_TITLE_FS      = 15
    LEGEND_FS           = 16
    CB_TICK_FS          = 13
    CB_LABEL_FS         = 15
    CLABEL_FS           = 12
    SPINE_W             = 1.0

    # === Overlay colors: first=light blue, second=orange ===
    OVERLAY_COLORS = ["#5BB8F5", "#F5882A"]

    # === Output file names ===
    base, ext = os.path.splitext(out_png)
    f_main = f"{base}__main{ext}"
    f_in1  = f"{base}__inset_1{ext}"
    f_in2  = f"{base}__inset_2{ext}"

    # === Rebase ===
    if rebase:
        y0, y1 = baseline_years
        df_models_plot = _rebase_df_like(df_models, y0, y1)
        mmm_plot = _rebase_series_like(mmm, y0, y1)
        overlays_plot = None
        if overlays:
            overlays_plot = {}
            for label, ser in overlays.items():
                overlays_plot[label] = _rebase_series_like(ser, y0, y1)
    else:
        df_models_plot = df_models
        mmm_plot = mmm
        overlays_plot = overlays

    # === Figure ===
    fig = plt.figure(figsize=(20.5, 7.8), constrained_layout=False)
    gs  = fig.add_gridspec(
        nrows=2, ncols=2,
        width_ratios=[1.6, 1.0],
        height_ratios=[1, 1],
        hspace=0.55,
        wspace=0.32,
    )

    ax   = fig.add_subplot(gs[:, 0])
    axR1 = fig.add_subplot(gs[0, 1])
    axR2 = fig.add_subplot(gs[1, 1])

    # === Main time series ===
    for mdl in df_models_plot.columns:
        ax.plot(df_models_plot.index, df_models_plot[mdl].values, lw=0.8, alpha=0.25)
    ax.plot(mmm_plot.index, mmm_plot.values, lw=2.6, label="Mean CMIP6", zorder=5, color="k")

    if overlays_plot:
        for (label, ser), color in zip(overlays_plot.items(), OVERLAY_COLORS):
            ser_plot = ser[ser.index.intersection(mmm_plot.index)]
            ax.plot(ser_plot.index, ser_plot.values, lw=1.8, label=label, zorder=6, color=color)

    ax.set_xlim(1900, 2030)
    ax.set_ylim(-4, 4)
    ax.set_xlabel("Year", fontsize=AXIS_LABEL_FS)
    ax.set_ylabel("AMOC 26.5°N anomaly (Sv)", fontsize=AXIS_LABEL_FS)
    ax.legend(frameon=False, fontsize=LEGEND_FS, loc="best")
    ax.tick_params(axis="both", labelsize=TICK_LABEL_FS_MAIN)
    for sp in ax.spines.values():
        sp.set_linewidth(SPINE_W)

    # === Color range for insets ===
    vmax = 2.0
    if anom_fields:
        mx = [np.nanmax(np.abs(f.values)) for f in anom_fields.values() if f is not None]
        if mx:
            vmax = float(np.ceil(np.nanmax(mx) * 2) / 2.0)

    pcm_global = None

    def draw_right_panel(iax, yr, x0, x1, *, show_xlabel=False):
        nonlocal pcm_global

        seg = mmm_plot.loc[x0:x1]
        if seg.size and np.isfinite(seg.values).any():
            y_min, y_max = float(np.nanmin(seg.values)), float(np.nanmax(seg.values))
        else:
            y_min, y_max = -0.2, 0.2
        pad = max(0.05, 0.05*(y_max - y_min))
        y0_box, y1_box = y_min - pad, y_max + pad
        ax.add_patch(Rectangle((x0, y0_box), x1 - x0, y1_box - y0_box,
                               fill=False, lw=1.1, ls="--", ec="k", alpha=0.5, zorder=2))

        fld = anom_fields.get(yr)
        if fld is not None:
            X, Y = np.meshgrid(fld["rlat"].values, fld["lev"].values)
            pcm = iax.pcolormesh(X, Y, fld.values, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto", zorder=5)
            if pcm_global is None:
                pcm_global = pcm

            try:
                iax.contour(X, Y, fld.values, levels=[0.0], colors="k", linewidths=0.8, zorder=6)
            except Exception:
                pass

            if pi_field is not None:
                pf = pi_field.interp(rlat=fld.rlat, lev=fld.lev)
                levels = (np.arange(5.0, max(10.0, np.ceil(np.nanmax(pf.values)/5.0)*5.0)+0.1, 5.0)
                          if pi_levels is None else pi_levels)
                cs = iax.contour(X, Y, pf.values, levels=levels,
                                 colors="k", linewidths=0.75, linestyles=":", zorder=6)
                try:
                    iax.clabel(cs, fmt="%.0f", inline=True, fontsize=CLABEL_FS)
                except Exception:
                    pass

            iax.axvline(26.0, lw=1.1, color="k", alpha=0.9, ls="--", zorder=7)
            iax.set_xlim(0, 60)
            iax.set_ylim(2000, 0)
            iax.set_xlabel("Latitude (°N)" if show_xlabel else "", fontsize=INSET_LABEL_FS)
            iax.set_ylabel("Depth (m)", fontsize=INSET_LABEL_FS)
            iax.set_xticks([0, 30, 60])
            iax.set_yticks([0, 1000, 2000])
            iax.tick_params(labelsize=TICK_LABEL_FS_INSET)
            iax.set_title(f"Δψ {x0}–{x1}", fontsize=INSET_TITLE_FS, pad=6)

            _draw_amoc_arrows(iax)

        else:
            iax.text(0.5, 0.5, f"No data {x0}-{x1}", ha="center", va="center", fontsize=12)
            iax.axis("off")

        # connector arrow from inset to main panel
        xyA = (0.0, 1.0)
        xyB = (0.5*(x0+x1), 0.5*(y0_box+y1_box))
        con = ConnectionPatch(
            xyA=xyA, coordsA=iax.transAxes,
            xyB=xyB, coordsB=ax.transData,
            arrowstyle="-|>", lw=1.8, alpha=0.65, color="gray",
            shrinkA=6, shrinkB=0, mutation_scale=16,
            zorder=10, clip_on=False,
        )
        fig.add_artist(con)
        for sp in iax.spines.values():
            sp.set_linewidth(SPINE_W)

    # === Draw both insets ===
    items = sorted(windows.items())
    if len(items) >= 1:
        yr, (x0, x1) = items[0]
        draw_right_panel(axR1, yr, x0, x1, show_xlabel=False)
    if len(items) >= 2:
        yr, (x0, x1) = items[1]
        draw_right_panel(axR2, yr, x0, x1, show_xlabel=True)

    # === Compact insets + single vertical colorbar ===
    shrink = 0.80
    p1 = axR1.get_position(fig)
    p2 = axR2.get_position(fig)
    axR1.set_position([p1.x0, p1.y0, p1.width*shrink, p1.height])
    axR2.set_position([p2.x0, p2.y0, p2.width*shrink, p2.height])

    p1 = axR1.get_position(fig)
    p2 = axR2.get_position(fig)
    y0_cb = min(p1.y0, p2.y0)
    y1_cb = max(p1.y1, p2.y1)
    right_edge = max(p1.x1, p2.x1)
    gap = 0.03
    cbar_w = 0.02
    pad_y = 0.02 * (y1_cb - y0_cb)
    cax = fig.add_axes([right_edge + gap, y0_cb + pad_y, cbar_w, (y1_cb - y0_cb) - 2*pad_y])

    if pcm_global is not None:
        cb = fig.colorbar(pcm_global, cax=cax, orientation="vertical")
        cb.set_label("Δψ (Sv)", fontsize=CB_LABEL_FS)
        cb.ax.tick_params(labelsize=CB_TICK_FS)

    # === Save combined figure ===
    fig.savefig(out_png, dpi=260, bbox_inches="tight")
    plt.close(fig)

    # === Save three panels separately ===
    if save_individual:
        fig_m, ax_m = plt.subplots(figsize=(8, 4.2))
        for mdl in df_models_plot.columns:
            ax_m.plot(df_models_plot.index, df_models_plot[mdl].values, lw=0.8, alpha=0.25)
        ax_m.plot(mmm_plot.index, mmm_plot.values, lw=2.6, label="Mean CMIP6", zorder=5, color="k")

        if overlays_plot:
            for (label, ser), color in zip(overlays_plot.items(), OVERLAY_COLORS):
                ser_plot = ser[ser.index.intersection(mmm_plot.index)]
                ax_m.plot(ser_plot.index, ser_plot.values, lw=1.8, label=label, zorder=6, color=color)

        ax_m.set_xlim(1900, 2030)
        ax_m.set_ylim(-4, 4)
        ax_m.set_xlabel("Year", fontsize=AXIS_LABEL_FS)
        ax_m.set_ylabel("AMOC 26.5°N anomaly (Sv, relative to 1900-1940 mean)", fontsize=AXIS_LABEL_FS)
        ax_m.tick_params(axis="both", labelsize=TICK_LABEL_FS_MAIN)
        ax_m.legend(frameon=False, fontsize=LEGEND_FS, loc="best")
        for sp in ax_m.spines.values():
            sp.set_linewidth(SPINE_W)
        fig_m.savefig(f_main, dpi=300, bbox_inches="tight")
        plt.close(fig_m)

        def save_inset(fpath, yr, x0, x1):
            fld = anom_fields.get(yr)
            fig_i, ax_i = plt.subplots(figsize=(6.0, 4.5))
            if fld is not None:
                X, Y = np.meshgrid(fld["rlat"].values, fld["lev"].values)
                pcm = ax_i.pcolormesh(X, Y, fld.values, cmap="RdBu_r",
                                      vmin=-vmax, vmax=vmax, shading="auto")
                try:
                    ax_i.contour(X, Y, fld.values, levels=[0.0], colors="k", linewidths=0.8)
                except Exception:
                    pass
                if pi_field is not None:
                    pf = pi_field.interp(rlat=fld.rlat, lev=fld.lev)
                    levels = (np.arange(5.0, max(10.0, np.ceil(np.nanmax(pf.values)/5.0)*5.0)+0.1, 5.0)
                              if pi_levels is None else pi_levels)
                    cs = ax_i.contour(X, Y, pf.values, levels=levels,
                                      colors="k", linewidths=0.75, linestyles=":")
                    try:
                        ax_i.clabel(cs, fmt="%.0f", inline=True, fontsize=CLABEL_FS)
                    except Exception:
                        pass

                ax_i.axvline(26.0, lw=1.1, color="k", alpha=0.9, ls="--")
                ax_i.set_xlim(0, 60)
                ax_i.set_ylim(2000, 0)
                ax_i.set_xlabel("Latitude (°N)", fontsize=INSET_LABEL_FS)
                ax_i.set_ylabel("Depth (m)", fontsize=INSET_LABEL_FS)
                ax_i.set_xticks([0, 30, 60])
                ax_i.set_yticks([0, 1000, 2000])
                ax_i.tick_params(labelsize=TICK_LABEL_FS_INSET)
                ax_i.set_title(f"Δψ {x0}–{x1}", fontsize=INSET_TITLE_FS)
                for sp in ax_i.spines.values():
                    sp.set_linewidth(SPINE_W)
                _draw_amoc_arrows(ax_i)
                cb = fig_i.colorbar(pcm, ax=ax_i, orientation="vertical", fraction=0.046, pad=0.04)
                cb.set_label("Δψ (Sv)", fontsize=CB_LABEL_FS)
                cb.ax.tick_params(labelsize=CB_TICK_FS)
            else:
                ax_i.text(0.5, 0.5, f"No data {x0}-{x1}", ha="center", va="center", fontsize=12)
                ax_i.axis("off")
            fig_i.savefig(fpath, dpi=300, bbox_inches="tight")
            plt.close(fig_i)

        if len(items) >= 1:
            yr, (x0, x1) = items[0]
            save_inset(f_in1, yr, x0, x1)
        if len(items) >= 2:
            yr, (x0, x1) = items[1]
            save_inset(f_in2, yr, x0, x1)

In [ ]:
plot_combo_external_insets_fullheight_onecb_vertical(
    df_models=df_models_anom,
    mmm=mmm_anom,
    anom_fields=anom_fields,
    windows=SNAP_WINDOWS,
    out_png=os.path.join(FIGS_DIR, "AMOC_MMM_anom_with_insets_combo_plus_HadISST.png"),
    pi_field=mmm_pi_field,
    overlays={"SST index (CMIP5)": proxy_series, "SST index (CMIP6)" : load_proxy_series_nc(PROXY_NC, factor=1.6, baseline=(1900, 1930))},
    #overlays={"SST index [1]": proxy_series},

)


